# NB25 — Hybrid GRU+FC with Direct AUC Optimization (LibAUC AUCM)

**Goal:** Replace `BCEWithLogitsLoss` in the NB21 Hybrid GRU+FC with LibAUC's compositional AUC maximization loss (AUCM). The model architecture is **identical** to NB21 — the only change is the training objective.

## Why BCE is Suboptimal for AUC

The competition metric is AUC-ROC, a **rank-ordering** metric. BCE optimizes log-likelihood — it penalizes wrong probability magnitudes. A model that assigns 0.6 to positive rows and 0.3 to negatives is perfect AUC (all positives ranked above negatives) but has non-zero BCE. AUCM directly minimizes an upper bound on 1-AUC via a compositional mini-max objective, guaranteeing that gradient steps always improve the rank ordering.

**AUCM loss (Yuan et al. 2021):**
```
L(a, b, α) = E_+[(f(x) - a - m)²] + E_-[(f(x) - b + m)²] + 2α(E_+[f(x) - a] - E_-[f(x) - b])
```
where `a`, `b` are running margin estimates, `α` is a dual variable, `m=1` is the margin. Minimizing over `(f, a, b)` and maximizing over `α` is equivalent to minimizing a tight upper bound on 1-AUC.

## Implementation

- Loss: `AUCMLoss(imratio=0.199, margin=1.0)` — re-initialized per fold (loss tracks running `a`, `b`, `α`)
- Optimizer: `AdamW(lr=5e-4)` + AUCM loss (PESG is optional — see `USE_PESG` flag)
- All other hyperparameters and architecture: **same as NB21**

## Expected Gain

LibAUC AUCM has shown +0.003–0.008 AUC gains on tabular benchmarks vs BCE. For our Hybrid (NB21 solo 0.9185 CV), expected target: **0.921–0.926 CV**, proj LB ~0.938–0.943.

If AUC ≥ 0.920: submit NB25 solo (safe bet to improve on 0.93618 LB).

## Inputs / Outputs
**Inputs:** `train_features_v2.parquet`, `test_features_v2.parquet`, `fold_assignments.parquet`,
`oof_predictions_nb21.parquet` (hybrid_oof for ρ comparison)

**Outputs:** `oof_predictions_nb25.parquet` (hybrid_aucm_oof), `test_predictions_nb25.parquet` (hybrid_aucm_pred),
`models/hybrid_aucm_fold{0-4}.pt`, `models/nb25_summary.pkl`

In [ ]:
# nb25-00  Install LibAUC
import subprocess
result = subprocess.run(['pip', 'install', 'libauc', '-q'], capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else 'No stdout')
if result.returncode != 0:
    print('INSTALL FAILED:', result.stderr[-500:])
else:
    import libauc
    print(f'LibAUC version: {libauc.__version__}')

In [ ]:
# nb25-01  Imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, rankdata
import pickle, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

try:
    from libauc.losses import AUCMLoss
    from libauc.optimizers import PESG
    LIBAUC_OK = True
    print('LibAUC imports OK')
except ImportError as e:
    print(f'LibAUC import failed: {e}')
    print('Will fallback to BCEWithLogitsLoss.')
    LIBAUC_OK = False

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# nb25-02  Path detection + data load
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root = cwd
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
models_dir.mkdir(exist_ok=True)
print(f'Root: {project_root}')

train      = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test       = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds_df   = pd.read_parquet(processed_dir / 'fold_assignments.parquet')
hybrid_oof = pd.read_parquet(processed_dir / 'oof_predictions_nb21.parquet')[['id', 'hybrid_oof']]

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Folds: {folds_df.fold.value_counts().sort_index().to_dict()}')

In [ ]:
# nb25-03  Settings  (identical to NB21 — only loss function changes)
SEQ_COLS = [
    'LapTime (s)', 'TyreLife', 'Cumulative_Degradation_winsorized',
    'LapTime_Delta', 'Position',
]

STRAT_COLS = [
    'Stint', 'RaceProgress', 'laps_remaining', 'compound_ordinal', 'is_wet_tyre',
    'prime_pit_window', 'laps_to_driver_end', 'abs_position_change',
    'pos_change_rolling_std_3', 'PitStop_lag1',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]
assert len(STRAT_COLS) == 13

CAT_COLS    = ['Driver', 'Race', 'Compound', 'Year']
WINDOW      = 10
IMRATIO     = 0.199     # positive rate in train (~20%) — for AUCMLoss
MARGIN      = 1.0       # AUCMLoss margin
BATCH_SIZE  = 2048
MAX_EPOCHS  = 80
PATIENCE    = 10
LR          = 5e-4      # AdamW lr (unused when USE_PESG=True)
WEIGHT_DECAY= 1e-4
N_FOLDS     = 5

# PESG is the correct optimizer for AUCM (handles primal-descent + dual-ascent on alpha).
# AdamW + criterion.parameters() was descending on alpha too — inverted the dual variable,
# causing AUC=0.27. PESG fixes this. USE_PESG=True is now the default.
USE_PESG    = True
PESG_LR     = 0.1       # PESG uses different lr scale than AdamW
EPOCH_DECAY = 2e-3      # PESG internal dual variable decay per epoch

print(f'LIBAUC_OK={LIBAUC_OK}, USE_PESG={USE_PESG}')
print(f'Loss: {"AUCMLoss(imratio=" + str(IMRATIO) + ",margin=" + str(MARGIN) + ")" if LIBAUC_OK else "BCEWithLogitsLoss(fallback)"}')
print(f'Optimizer: {"PESG(lr=" + str(PESG_LR) + ",epoch_decay=" + str(EPOCH_DECAY) + ")" if (LIBAUC_OK and USE_PESG) else "AdamW(lr=" + str(LR) + ")"}')
if DEVICE.type == 'cpu':
    print('WARNING: CPU-only. Upload to Kaggle T4.')

In [ ]:
# nb25-04  Label encoders for entity embeddings (fit on combined train+test)
combined_cats = pd.concat([train[CAT_COLS], test[CAT_COLS]], ignore_index=True)
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(combined_cats[col].astype(str))
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} classes')

for col in CAT_COLS:
    le = label_encoders[col]
    train[col + '_idx'] = le.transform(train[col].astype(str))
    test[col  + '_idx'] = le.transform(test[col].astype(str))

EMB_DIMS = {
    'Driver':   (len(label_encoders['Driver'].classes_)   + 1, 32),
    'Race':     (len(label_encoders['Race'].classes_)     + 1,  8),
    'Compound': (len(label_encoders['Compound'].classes_) + 1,  3),
    'Year':     (len(label_encoders['Year'].classes_)     + 1,  2),
}
print('Embedding dims:', EMB_DIMS)

In [ ]:
# nb25-05  Build 10-lap sequence windows  (identical to NB21)
idx_cols     = ['id', 'Race', 'Year', 'Driver', 'LapNumber']
cat_idx_cols = [c + '_idx' for c in CAT_COLS]

seq_scaler = StandardScaler()
seq_scaler.fit(train[SEQ_COLS].values)

all_sel = list(dict.fromkeys(idx_cols + SEQ_COLS + STRAT_COLS + cat_idx_cols))

combined_df = pd.concat([
    train[all_sel + ['PitNextLap']].assign(is_train=True),
    test[ all_sel].assign(is_train=False, PitNextLap=-1),
], ignore_index=True)

combined_df = combined_df.sort_values(['Race', 'Year', 'Driver', 'LapNumber']).reset_index(drop=True)
combined_df[SEQ_COLS] = seq_scaler.transform(combined_df[SEQ_COLS].values).astype(np.float32)

N        = len(combined_df)
N_SF     = len(SEQ_COLS)
seq_vals = combined_df[SEQ_COLS].values.astype(np.float32)

all_windows = np.zeros((N, WINDOW, N_SF), dtype=np.float32)
all_masks   = np.zeros((N, WINDOW),       dtype=bool)

GROUP_COLS = ['Race', 'Year', 'Driver']
print(f'Building windows for {N:,} rows  (window={WINDOW}, features={N_SF})...')
t0 = time.time()

for _, grp_idx in tqdm(combined_df.groupby(GROUP_COLS, sort=False).groups.items(),
                       total=combined_df[GROUP_COLS].drop_duplicates().shape[0]):
    idxs  = grp_idx.values
    n_grp = len(idxs)
    for i in range(n_grp):
        hist_len = min(i, WINDOW)
        if hist_len > 0:
            all_windows[idxs[i], WINDOW - hist_len:] = seq_vals[idxs[i - hist_len : i]]
            all_masks[idxs[i],   WINDOW - hist_len:] = True

print(f'Done in {time.time()-t0:.1f}s')

In [ ]:
# nb25-06  Split back to train / test arrays  (identical to NB21)
tr_mask = combined_df['is_train'].values
te_mask = ~tr_mask

train_windows   = all_windows[tr_mask]
train_seq_masks = all_masks[tr_mask]
test_windows    = all_windows[te_mask]
test_seq_masks  = all_masks[te_mask]

train_strat_raw = combined_df.loc[tr_mask, STRAT_COLS].values.astype(np.float32)
test_strat_raw  = combined_df.loc[te_mask, STRAT_COLS].values.astype(np.float32)

train_targets = combined_df.loc[tr_mask, 'PitNextLap'].values.astype(np.float32)
train_ids     = combined_df.loc[tr_mask, 'id'].values
test_ids      = combined_df.loc[te_mask, 'id'].values

fold_lookup  = folds_df.set_index('id')['fold']
train_folds  = fold_lookup.loc[train_ids].values

train_cat = {c: combined_df.loc[tr_mask, c+'_idx'].values for c in CAT_COLS}
test_cat  = {c: combined_df.loc[te_mask, c+'_idx'].values for c in CAT_COLS}

del all_windows, all_masks, combined_df
print(f'Train: {train_windows.shape}, Test: {test_windows.shape}')
print(f'Fold counts: {pd.Series(train_folds).value_counts().sort_index().to_dict()}')

In [ ]:
# nb25-07  Model  (IDENTICAL to NB21 HybridGRUFC — no architecture changes)
class HybridGRUFC(nn.Module):
    """Dual-branch: GRU(10-lap tyre window) + FC(13 strategic scalars) + entity embeddings."""

    def __init__(self, n_strat=13, n_seq=5, gru_hidden=128, n_layers=2, gru_drop=0.15,
                 emb_dims=None):
        super().__init__()
        if emb_dims is None:
            emb_dims = EMB_DIMS

        self.driver_emb   = nn.Embedding(*emb_dims['Driver'])
        self.race_emb     = nn.Embedding(*emb_dims['Race'])
        self.compound_emb = nn.Embedding(*emb_dims['Compound'])
        self.year_emb     = nn.Embedding(*emb_dims['Year'])
        emb_total = sum(v[1] for v in emb_dims.values())  # 45

        self.gru = nn.GRU(
            input_size=n_seq,
            hidden_size=gru_hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=gru_drop if n_layers > 1 else 0.0,
        )

        self.strat_fc = nn.Sequential(
            nn.Linear(n_strat, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )

        merge_dim = gru_hidden + 64 + emb_total  # 237
        self.head = nn.Sequential(
            nn.Linear(merge_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )

    def forward(self, window, mask, strat, driver, race, compound, year):
        emb = torch.cat([
            self.driver_emb(driver), self.race_emb(race),
            self.compound_emb(compound), self.year_emb(year),
        ], dim=1)

        seq_len  = mask.sum(dim=1).long().clamp(min=1)
        packed   = nn.utils.rnn.pack_padded_sequence(
            window, seq_len.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n   = self.gru(packed)
        gru_feat = h_n[-1]  # (B, 128)

        strat_feat = self.strat_fc(strat)  # (B, 64)

        merged = torch.cat([gru_feat, strat_feat, emb], dim=1)  # (B, 237)
        return self.head(merged).squeeze(1)  # raw logit (B,)


_m = HybridGRUFC()
print(f'Model params: {sum(p.numel() for p in _m.parameters()):,}')
del _m

In [ ]:
# nb25-08  Dataset + DataLoader  (identical to NB21)
class F1HybridDataset(Dataset):
    def __init__(self, windows, masks, strat, cat_idxs, targets=None):
        self.windows  = torch.tensor(windows, dtype=torch.float32)
        self.masks    = torch.tensor(masks,   dtype=torch.bool)
        self.strat    = torch.tensor(strat,   dtype=torch.float32)
        self.driver   = torch.tensor(cat_idxs['Driver'],   dtype=torch.long)
        self.race     = torch.tensor(cat_idxs['Race'],     dtype=torch.long)
        self.compound = torch.tensor(cat_idxs['Compound'], dtype=torch.long)
        self.year     = torch.tensor(cat_idxs['Year'],     dtype=torch.long)
        self.targets  = (None if targets is None
                         else torch.tensor(targets, dtype=torch.float32))

    def __len__(self):  return len(self.windows)

    def __getitem__(self, i):
        d = dict(window=self.windows[i], mask=self.masks[i], strat=self.strat[i],
                 driver=self.driver[i], race=self.race[i],
                 compound=self.compound[i], year=self.year[i])
        if self.targets is not None:
            d['target'] = self.targets[i]
        return d


def make_loader(windows, masks, strat, cat_idxs, targets=None, shuffle=False):
    ds = F1HybridDataset(windows, masks, strat, cat_idxs, targets)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

In [ ]:
# nb25-09  Training utilities
# KEY CHANGE vs NB21: criterion is AUCMLoss (not BCE). AUCMLoss expects probabilities
# (sigmoid outputs), not raw logits. Apply sigmoid inside training loop.
# Inference: still return sigmoid(logit) probabilities for AUC evaluation.

def run_epoch(model, loader, criterion=None, optimizer=None, use_aucm=False):
    """Run one epoch. criterion=None → inference-only (no loss computed)."""
    is_train = optimizer is not None and criterion is not None
    model.train(is_train)
    total_loss, all_probs = 0.0, []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            win  = batch['window'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            st   = batch['strat'].to(DEVICE)
            drv  = batch['driver'].to(DEVICE)
            rc   = batch['race'].to(DEVICE)
            cmp  = batch['compound'].to(DEVICE)
            yr   = batch['year'].to(DEVICE)

            logits = model(win, mask, st, drv, rc, cmp, yr)  # raw logits
            probs  = torch.sigmoid(logits)                   # probabilities

            if is_train:
                tgt  = batch['target'].to(DEVICE)
                if use_aucm:
                    # AUCMLoss expects probabilities and 0/1 labels
                    loss = criterion(probs, tgt)
                else:
                    # BCE fallback uses logits
                    loss = criterion(logits, tgt)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item() * len(win)

            all_probs.append(probs.detach().cpu().float().numpy())

    return np.concatenate(all_probs), total_loss / max(len(loader.dataset), 1)

In [ ]:
# nb25-10  5-fold CV training with AUCMLoss + PESG
# KEY CHANGES vs NB21:
#   1. Loss per fold: AUCMLoss(imratio=IMRATIO) re-initialized each fold
#      (AUCM tracks running a, b, alpha — must be fresh per fold)
#   2. Optimizer: PESG (primal-dual saddle-point) — NOT AdamW with criterion.parameters().
#      AdamW descended on the Lagrange multiplier alpha (should be ascended), inverting
#      the dual update and causing AUC=0.27. PESG handles primal/dual correctly internally.
#   3. No external LR scheduler for PESG — it uses epoch_decay internally. CosineAnnealingLR
#      on top of PESG interferes with the saddle-point dynamics.
#   4. Training step: pass probabilities to AUCM, logits to BCE fallback.

oof_preds        = np.zeros(len(train_ids))
test_preds_folds = []
fold_aucs        = []
best_epochs      = []
use_aucm         = LIBAUC_OK

for fold in range(N_FOLDS):
    print(f'\n{"="*60}')
    print(f'Fold {fold}  [loss: {"AUCM+PESG" if (use_aucm and USE_PESG) else "AUCM+AdamW" if use_aucm else "BCE-fallback"}]')
    print('='*60)

    tr_idx = np.where(train_folds != fold)[0]
    va_idx = np.where(train_folds == fold)[0]

    strat_scaler = StandardScaler()
    tr_strat = strat_scaler.fit_transform(train_strat_raw[tr_idx])
    va_strat = strat_scaler.transform(train_strat_raw[va_idx])
    te_strat = strat_scaler.transform(test_strat_raw)

    tr_cat = {c: train_cat[c][tr_idx] for c in CAT_COLS}
    va_cat = {c: train_cat[c][va_idx] for c in CAT_COLS}

    tr_loader = make_loader(train_windows[tr_idx], train_seq_masks[tr_idx],
                            tr_strat, tr_cat, targets=train_targets[tr_idx], shuffle=True)
    va_loader = make_loader(train_windows[va_idx], train_seq_masks[va_idx], va_strat, va_cat)
    te_loader = make_loader(test_windows, test_seq_masks, te_strat, test_cat)

    model = HybridGRUFC().to(DEVICE)

    # --- Loss and optimizer setup ---
    if use_aucm:
        # Re-initialize per fold so internal a, b, alpha start fresh
        criterion = AUCMLoss(imratio=IMRATIO, margin=MARGIN).to(DEVICE)
        if USE_PESG:
            # PESG: designed for AUCM saddle-point (primal descent on model+a+b, dual ascent on alpha).
            # Do NOT add a CosineAnnealingLR — PESG schedules itself via epoch_decay.
            optimizer = PESG(
                model.parameters(),
                loss_fn=criterion,
                lr=PESG_LR,
                epoch_decay=EPOCH_DECAY,
                weight_decay=WEIGHT_DECAY,
                momentum=0.9,
                device=DEVICE,
            )
            scheduler = None  # PESG uses epoch_decay internally — no external scheduler
        else:
            # AdamW fallback (kept for reference — known to fail due to alpha inversion)
            optimizer = torch.optim.AdamW(
                model.parameters(),  # criterion.parameters() intentionally excluded
                lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    else:
        # BCE fallback if LibAUC unavailable
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([1.0 / IMRATIO - 1.0], device=DEVICE))
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    best_auc, best_epoch, patience_ctr, best_state = 0.0, 0, 0, None

    for epoch in range(MAX_EPOCHS):
        _, tr_loss   = run_epoch(model, tr_loader, criterion, optimizer, use_aucm=use_aucm)
        if scheduler is not None:
            scheduler.step()
        val_probs, _ = run_epoch(model, va_loader)
        val_auc      = roc_auc_score(train_targets[va_idx], val_probs)

        if val_auc > best_auc:
            best_auc, best_epoch, patience_ctr = val_auc, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if (epoch + 1) % 10 == 0 or patience_ctr >= PATIENCE:
            print(f'  ep {epoch+1:3d}  loss={tr_loss:.4f}  val={val_auc:.4f}  '
                  f'best={best_auc:.4f} (ep{best_epoch+1})')

        if patience_ctr >= PATIENCE:
            print('  Early stop')
            break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    oof_probs, _ = run_epoch(model, va_loader)
    oof_preds[va_idx] = oof_probs
    fold_auc = roc_auc_score(train_targets[va_idx], oof_probs)
    fold_aucs.append(fold_auc)
    best_epochs.append(best_epoch + 1)
    print(f'  Fold {fold} AUC: {fold_auc:.4f}  (best ep {best_epoch+1})')

    te_probs, _ = run_epoch(model, te_loader)
    test_preds_folds.append(te_probs)

    torch.save(best_state, models_dir / f'hybrid_aucm_fold{fold}.pt')
    print(f'  Saved hybrid_aucm_fold{fold}.pt')

oof_auc = roc_auc_score(train_targets, oof_preds)
print(f'\nOOF AUC : {oof_auc:.4f}')
print(f'Fold AUCs: {[f"{a:.4f}" for a in fold_aucs]}')
print(f'Fold std : {np.std(fold_aucs):.4f}')
print(f'Best eps : {best_epochs}')

In [ ]:
# nb25-11  Correlation analysis + gate check vs NB21 Hybrid and LGBM
nb21_vals  = hybrid_oof.set_index('id').loc[train_ids, 'hybrid_oof'].values
lgbm_df    = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')
lgbm_vals  = lgbm_df.set_index('id').loc[train_ids, 'lgbm_tuned_oof'].values

rho_nb21, _ = spearmanr(oof_preds, nb21_vals)
rho_lgbm, _ = spearmanr(oof_preds, lgbm_vals)

nb21_auc = roc_auc_score(train_targets, nb21_vals)
lgbm_auc = roc_auc_score(train_targets, lgbm_vals)

print(f'NB25 AUCM solo OOF AUC : {oof_auc:.4f}')
print(f'NB21 BCE  solo OOF AUC : {nb21_auc:.4f}  (baseline to beat)')
print(f'LGBM NB12 solo OOF AUC : {lgbm_auc:.4f}')
print(f'Delta vs NB21          : {oof_auc - nb21_auc:+.4f}')
print()
print('Spearman rho:')
print(f'  NB25 x NB21 : {rho_nb21:.4f}  (expect ~0.95+ — same arch, diff loss)')
print(f'  NB25 x LGBM : {rho_lgbm:.4f}')

# Decision: is NB25 better than NB21?
print()
if oof_auc >= 0.920:
    print('DECISION: NB25 AUC >= 0.920. Consider solo submission.')
    print(f'  Proj LB (solo, +0.0177 boost): {oof_auc + 0.0177:.4f}')
elif oof_auc > nb21_auc:
    print(f'DECISION: NB25 (+{oof_auc - nb21_auc:.4f} vs NB21). Too small to submit solo.')
    print('  Consider replacing NB21 in the 3-seed rank avg if NB25 > 0.919.')
else:
    print(f'DECISION: NB25 ({oof_auc:.4f}) <= NB21 ({nb21_auc:.4f}). AUCM did not help. Do not submit.')
    if not use_aucm:
        print('  (Note: BCE fallback was used — LibAUC was not available)')

# Rank-avg preview with NB21
def rank_norm(a): return rankdata(a) / len(a)
ens = roc_auc_score(train_targets, (rank_norm(oof_preds) + rank_norm(nb21_vals)) / 2)
print(f'\nRank avg NB25+NB21: {ens:.4f}  (note: high rho={rho_nb21:.4f}, gain likely noise)')

In [ ]:
# nb25-12  Average test predictions across folds
test_preds_mean = np.mean(np.stack(test_preds_folds, axis=0), axis=0)
print(f'Test preds: shape={test_preds_mean.shape}, mean={test_preds_mean.mean():.4f}, '
      f'std={test_preds_mean.std():.4f}')

In [ ]:
# nb25-13  Save outputs
oof_df = pd.DataFrame({
    'id':                train_ids,
    'fold':              train_folds,
    'PitNextLap':        train_targets.astype(int),
    'hybrid_aucm_oof':   oof_preds,
})
assert len(oof_df) == 439140
oof_df.to_parquet(processed_dir / 'oof_predictions_nb25.parquet', index=False)
print(f'Saved oof_predictions_nb25.parquet')

test_out = pd.DataFrame({'id': test_ids, 'hybrid_aucm_pred': test_preds_mean})
test_out.to_parquet(processed_dir / 'test_predictions_nb25.parquet', index=False)
print(f'Saved test_predictions_nb25.parquet')

summary = {
    'oof_auc':            oof_auc,
    'fold_aucs':          fold_aucs,
    'fold_std':           float(np.std(fold_aucs)),
    'best_epochs':        best_epochs,
    'delta_vs_nb21':      float(oof_auc - nb21_auc),
    'rho_vs_nb21':        float(rho_nb21),
    'rho_vs_lgbm':        float(rho_lgbm),
    'loss':               'AUCMLoss' if use_aucm else 'BCEWithLogitsLoss-fallback',
    'optimizer':          'PESG' if (use_aucm and USE_PESG) else 'AdamW',
    'model':              'HybridGRUFC-NB25-AUCM',
    'libauc_available':   LIBAUC_OK,
}
with open(models_dir / 'nb25_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)

print(f'\n=== NB25 Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
# nb25-14  Submission (if NB25 beats NB21 solo)
if oof_auc > nb21_auc + 0.001:  # meaningful improvement threshold
    sub = pd.DataFrame({'id': test_ids, 'PitNextLap': test_preds_mean})
    sub_path = project_root / 'submissions' / f'submission_v013_hybrid_aucm_solo_cv{oof_auc:.4f}.csv'
    sub.to_csv(sub_path, index=False)
    print(f'Submission saved: {sub_path}')
    print(f'CV OOF AUC: {oof_auc:.4f}')
    print(f'Projected LB (solo +0.0177 boost): {oof_auc + 0.0177:.4f}')
    print(f'Projected LB (3-seed if replacing NB21 seeds, +0.0142 boost): {oof_auc + 0.0142:.4f}')
else:
    print(f'NB25 AUC ({oof_auc:.4f}) not significantly better than NB21 ({nb21_auc:.4f}).')
    print('Do not submit NB25. AUCM loss did not provide meaningful gain.')
    print('Recommendation: proceed with NB24 TFT fixed results.')